# ML Zoomcamp 2026 — Homework 3: Classification

AI-assisted solution prepared for Nikita Galkin. Calculations executed against the official pinned 2026 dataset. This notebook is a worked learning example; independent mastery has not been assessed.

Source: https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/03-classification/homework.md

Target: `converted`. Use only training data to fit encoders/models and compute mutual information; validation data for model comparison. Test data remains untouched.

In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
print("pandas:", pd.__version__, "scikit-learn:", sklearn.__version__)
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)
print("Shape:", df.shape)
print(df.isna().sum())
categorical = df.select_dtypes(include="object").columns.tolist()
numerical = [c for c in df.select_dtypes(include="number").columns if c != "converted"]
df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)
assert df.converted.notna().all()
assert not df.isna().any().any()
features = categorical + numerical

pandas: 2.2.3 scikit-learn: 1.8.0
Shape: (5000, 9)
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


## Q1 — Mode
Mode means the most frequent value. Missing industries are filled with the literal string `NA` first.

In [2]:
print(df.industry.value_counts())
q1 = df.industry.mode().iloc[0]
print("Q1:", q1)

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64
Q1: technology


## Q2 — Correlation
Compare only the four pairs named in the assignment, using the full prepared dataset.

In [3]:
corr = df[numerical].corr()
display(corr)
pairs = [("interaction_count", "lead_score"), ("number_of_courses_viewed", "lead_score"), ("number_of_courses_viewed", "interaction_count"), ("annual_income", "interaction_count")]
pair_scores = {f"{a} and {b}": corr.loc[a,b] for a,b in pairs}
print(pair_scores)
q2 = max(pair_scores, key=pair_scores.get)
print("Q2:", q2)

                          annual_income  number_of_courses_viewed  interaction_count  lead_score
annual_income                  1.000000                  0.161300           0.122842    0.229496
number_of_courses_viewed       0.161300                  1.000000           0.721609    0.757204
interaction_count              0.122842                  0.721609           1.000000    0.915746
lead_score                     0.229496                  0.757204           0.915746    1.000000
{'interaction_count and lead_score': np.float64(0.9157457980418697), 'number_of_courses_viewed and lead_score': np.float64(0.7572042392104886), 'number_of_courses_viewed and interaction_count': np.float64(0.7216090038937035), 'annual_income and interaction_count': np.float64(0.12284235135955959)}
Q2: interaction_count and lead_score


## Split — 60% train / 20% validation / 20% test
Use the exact two calls from the assignment. Remove the target from predictor tables to prevent leakage.

In [4]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
y_train = df_train.converted.to_numpy()
y_val = df_val.converted.to_numpy()
y_test = df_test.converted.to_numpy()
df_train = df_train.drop(columns="converted").reset_index(drop=True)
df_val = df_val.drop(columns="converted").reset_index(drop=True)
df_test = df_test.drop(columns="converted").reset_index(drop=True)
assert "converted" not in features
print(len(df_train), len(df_val), len(df_test))

3000 1000 1000


## Q3 — Mutual information
Measure dependence between each categorical feature and the target using training data only; round to two decimals.

In [5]:
mi_raw = {c: mutual_info_score(df_train[c], y_train) for c in categorical}
mi_rounded = {c: round(v,2) for c,v in mi_raw.items()}
print("Raw:", mi_raw)
print("Rounded:", mi_rounded)
q3 = max(mi_rounded, key=mi_rounded.get)
print("Q3:", q3)

Raw: {'lead_source': 0.03434605791617973, 'industry': 0.0019945636715935217, 'employment_status': 0.018965604906982875, 'location': 0.001042763579701525}
Rounded: {'lead_source': 0.03, 'industry': 0.0, 'employment_status': 0.02, 'location': 0.0}
Q3: lead_source


## Q4 — Logistic regression
DictVectorizer one-hot encodes strings and preserves numeric values. Fit the vectorizer on training data only. Classification uses a 0.5 probability threshold.

In [6]:
def fit_score(selected_features, C=1.0):
    dv = DictVectorizer(sparse=True)
    X_train = dv.fit_transform(df_train[selected_features].to_dict(orient="records"))
    X_val = dv.transform(df_val[selected_features].to_dict(orient="records"))
    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    prediction = model.predict_proba(X_val)[:,1] >= 0.5
    return accuracy_score(y_val, prediction)
base_accuracy = fit_score(features)
q4 = round(base_accuracy,2)
print("Raw accuracy:", base_accuracy)
print("Q4:", q4)

Raw accuracy: 0.645
Q4: 0.65


## Q5 — Feature elimination
Drop each original feature separately and refit the encoder/model. Difference is baseline minus reduced-model accuracy; do not take its absolute value. Select among the three given options.

In [7]:
elimination = []
for feature in features:
    score = fit_score([c for c in features if c != feature])
    elimination.append({"feature": feature, "accuracy_without": score, "difference": base_accuracy-score})
elimination = pd.DataFrame(elimination)
display(elimination.sort_values("difference"))
q5_options = ["lead_source", "number_of_courses_viewed", "interaction_count"]
q5 = elimination[elimination.feature.isin(q5_options)].sort_values("difference").iloc[0].feature
print("Q5:", q5)

                    feature  accuracy_without  difference
4             annual_income             0.724      -0.079
1                  industry             0.645       0.000
3                  location             0.645       0.000
2         employment_status             0.645       0.000
7                lead_score             0.644       0.001
5  number_of_courses_viewed             0.643       0.002
0               lead_source             0.642       0.003
6         interaction_count             0.601       0.044
Q5: number_of_courses_viewed


## Q6 — Regularization
Smaller C means stronger regularization. Compare validation accuracies rounded to three decimals, then choose the smallest C in a tie.

In [8]:
tuning = []
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    score = fit_score(features,C)
    tuning.append({"C": C, "accuracy": score, "rounded_accuracy": round(score,3)})
tuning = pd.DataFrame(tuning)
display(tuning)
best = tuning.rounded_accuracy.max()
q6 = tuning.loc[tuning.rounded_accuracy == best,"C"].min()
print("Q6:", q6)
print("ANSWERS", {"Q1":q1,"Q2":q2,"Q3":q3,"Q4":q4,"Q5":q5,"Q6":q6})

          C  accuracy  rounded_accuracy
0  0.000001     0.598             0.598
1  0.000010     0.598             0.598
2  0.000100     0.613             0.613
3  0.001000     0.645             0.645
Q6: 0.001
ANSWERS {'Q1': 'technology', 'Q2': 'interaction_count and lead_score', 'Q3': 'lead_source', 'Q4': 0.65, 'Q5': 'number_of_courses_viewed', 'Q6': np.float64(0.001)}
